# The Polyglot's Shorthand — PS 4

Inter IIT Bootcamp 2026 · CS/AI Practice Problem Statements · **Problem 4 (450 pts)**

An ultra-efficient linguistic engine that treats **Latinized / Romanized code-mixed text as a
first-class citizen** — not as corrupted English, not as shattered Devanagari.

## Mapping to the problem statement

| PS requirement | Where it's addressed |
|---|---|
| Footprint: **max 500M params**, single-digit ms latency, high throughput | §4 footprint report (~15M params), §8 latency + throughput benchmark |
| Robustness: invariant to phonetic spelling drift, slang, typos | §1 canonicalization + consonant-skeleton channel, §6 drift augmentation, §7 perturbation sweep |
| Multimodal text semantics: emojis + punctuation as semantic modulators | §1 segmentation, §3 atomic emoji tokens, §7 emoji ablation |
| Deliverable: reproducible training **and inference** codebase | seeded end-to-end run, `config.json` + weights + tokenizer exported, `classify()` inference entry point in §9 |
| Deliverable: whitepaper — tokenization/architecture justification | §3 fertility measurement, §4 architecture rationale |
| Deliverable: whitepaper — **error analyses** | §7 sliced error analysis: code-mix ratio, length, emoji, fertility, ambiguous-word LID |
| Deliverable: whitepaper — **throughput trade-offs** | §8 batch sweep, sequence-length sweep, int8 accuracy-vs-latency trade-off |
| Note: *"any one/multiple downstream NLP tasks like Sentiment Analysis / Text Classification"* | sentence-level **sentiment** + token-level **language identification** |

Summarization and QA are deliberately out of scope — the Note permits choosing tasks, and no real
labelled Romanized Hinglish corpus exists for either. Inventing one would make those numbers
meaningless. Both are noted as extensions in §10.

## Datasets — real, Hinglish, downloaded automatically

| Corpus | Task | Size |
|---|---|---|
| **SemEval-2020 Task 9 — SentiMix Hinglish** | sentence sentiment (pos/neg/neu) + word-level language tags | 15,130 train / 3,000 labelled val / 3,000 unlabelled test |
| **L3Cube-HingLID** | word-level Hindi/English identification on Romanized Hinglish | 31,756 / 6,279 / 6,420 sentences |

SentiMix has a **published leaderboard** (best system 75.0 weighted F1), so our number lands in a
comparable frame. HingLID forces the encoder to model code-switching at the token level instead of
memorising sentence-level cues, and its tags also drive the code-mix-ratio slice in the error
analysis.

> **Kaggle internet must be ON** (Settings → Internet → On) for the download cell. If it's off,
> attach the files as a Kaggle Dataset — the loader scans `/kaggle/input` first and prints exactly
> which filenames it accepts.

> Set `SMOKE_TEST = True` for a ~3 minute sanity run before committing to the full run.

In [ ]:
# =====================================================================
# CONFIG  -- the only cell you normally need to touch
# =====================================================================
from dataclasses import dataclass

SMOKE_TEST = False          # True -> tiny run to verify the whole pipeline executes


@dataclass
class CFG:
    seed: int = 42

    # ---- tasks ----------------------------------------------------
    use_sent: bool = True        # SemEval-2020 Task 9 Hinglish sentiment  (headline metric)
    use_lid: bool = True         # word-level language identification

    # ---- preprocessing --------------------------------------------
    max_len: int = 96
    collapse_urls_mentions: bool = True   # "@ handle" -> <user>, "https // t . co / x" -> <url>
    augment_drift_p: float = 0.30         # fraction of TRAIN rows re-spelled each epoch
    augment_drift_strength: int = 1

    # ---- tokenizer -------------------------------------------------
    vocab_size: int = 16000
    skeleton_buckets: int = 8192

    # ---- model -----------------------------------------------------
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    d_ff: int = 1152
    dropout: float = 0.1

    # ---- MLM pretraining -------------------------------------------
    mlm_epochs: int = 14
    mlm_bs: int = 256
    mlm_lr: float = 3e-4
    mask_prob: float = 0.15

    # ---- multi-task fine-tuning ------------------------------------
    ft_epochs: int = 6
    ft_bs: int = 64
    ft_lr: float = 1e-4
    task_weights: tuple = (("sent", 1.0), ("lid", 0.5))

    # ---- runtime ---------------------------------------------------
    amp: bool = True
    out_dir: str = "/kaggle/working/polyglot"


cfg = CFG()

if SMOKE_TEST:
    cfg.vocab_size = 4000
    cfg.d_model, cfg.n_layers, cfg.n_heads, cfg.d_ff = 192, 2, 4, 576
    cfg.mlm_epochs, cfg.ft_epochs = 1, 1

cfg

In [ ]:
# =====================================================================
# ENVIRONMENT
# =====================================================================
import os, re, json, math, time, random, zlib, shutil, unicodedata, warnings, urllib.request
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

if not os.path.isdir("/kaggle/working"):
    cfg.out_dir = os.path.join(os.getcwd(), "polyglot")   # runs fine off-Kaggle too
DATA_DIR = os.path.join(cfg.out_dir, "data")
os.makedirs(DATA_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    cfg.mlm_epochs = min(cfg.mlm_epochs, 2)
    cfg.ft_epochs = min(cfg.ft_epochs, 2)
    cfg.amp = False


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


seed_all(cfg.seed)


def make_scaler(enabled):
    # torch.amp.GradScaler is the current API; fall back for older builds
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_ctx(enabled):
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=enabled)


print(f"torch   : {torch.__version__}")
print(f"device  : {DEVICE}  ({torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'cpu'})")
print(f"out_dir : {cfg.out_dir}")

---
## 1 · Normalization layer

Deterministic regex/string ops — no model calls, microseconds per sentence. This is the layer that
answers the PS's *"phonetic volatility & typos"* and *"pragmatic flips via emojis"* bullets.

1. **Segmentation** — emojis and punctuation split into standalone tokens so they reach the
   vocabulary instead of being glued to neighbouring words.
2. **Canonicalization** — light phonetic folding of the most common Romanization choices
   (`aa→a`, `ee→i`, `ph→f`, `z→j`, `w→v`, `q→k`, run-length capping). Shrinks the surface space
   the tokenizer must cover without destroying word identity.
3. **Consonant skeleton** — an aggressive key (first char + non-initial consonants, capped at 8)
   used as a *second input channel*. `kar` / `kr` / `karr` / `kaar` all hash to `kr`, so
   vowel-dropping shorthand shares an embedding. Short function words (`ki`, `ka`, `ko`) keep
   their own identity so the fold doesn't over-merge.
4. **Noise collapse** — `@ handle` → `<user>`, `https // t . co / abc123` → `<url>`. In SentiMix
   train that's ~21.9k and ~8.5k tokens of pure vocabulary noise.

In [ ]:
# =====================================================================
# NORMALIZATION
# =====================================================================
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U0001F000-\U0001F2FF"
    "\U00002600-\U000027BF"
    "\U00002B00-\U00002BFF"
    "\U0001F1E6-\U0001F1FF"
    "\U0000FE0F\U0000200D\U00002049\U0000203C\U00002190-\U000021FF"
    "]+"
)
PUNCT_RE = re.compile(r"([!?.,;:()\[\]\"'/@#%&*+=<>~|{}$-])")
WS_RE = re.compile(r"\s+")

DIGRAPHS = [
    ("aa", "a"), ("ii", "i"), ("ee", "i"), ("oo", "u"), ("uu", "u"),
    ("ph", "f"), ("ck", "k"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
    ("dh", "d"), ("th", "t"), ("jh", "j"), ("sh", "s"), ("ch", "c"),
    ("qu", "k"), ("ai", "e"), ("ay", "e"),
]
SINGLE_MAP = str.maketrans({"z": "j", "w": "v", "q": "k", "x": "ks", "y": "i"})
VOWELS = set("aeiou")
RUN_RE = re.compile(r"(.)\1{1,}")
ELONG_RE = re.compile(r"(.)\1{2,}")
PLACEHOLDERS = ("<user>", "<url>", "<num>")


def is_emoji(tok):
    return bool(tok) and EMOJI_RE.fullmatch(tok) is not None


def is_punct(tok):
    def _p(c):
        cat = unicodedata.category(c)
        return cat.startswith("P") or (cat.startswith("S") and not is_emoji(c))
    return bool(tok) and all(_p(c) for c in tok)


def segment(text):
    # split emojis / punctuation into their own tokens, normalize whitespace
    text = unicodedata.normalize("NFKC", str(text))
    text = EMOJI_RE.sub(lambda m: " " + m.group(0) + " ", text)
    text = PUNCT_RE.sub(r" \1 ", text)
    return WS_RE.sub(" ", text).strip().split()


def canonical(word):
    # light phonetic folding -- the tokenizer's surface form
    if word in PLACEHOLDERS:
        return word
    if is_emoji(word):
        return RUN_RE.sub(r"\1\1", word)      # 😂😂😂😂 -> 😂😂
    if not word.isalpha():
        return word.lower()
    w = word.lower()
    w = RUN_RE.sub(r"\1\1", w)                # cap any char run at 2
    for a, b in DIGRAPHS:
        w = w.replace(a, b)
    w = w.translate(SINGLE_MAP)
    w = RUN_RE.sub(r"\1", w)                  # collapse what the folding produced
    return w or word.lower()


def skeleton(word):
    # aggressive consonant key -- the drift-invariant channel
    if word in PLACEHOLDERS:
        return word
    if is_emoji(word):
        return "<emo>"
    if not word.isalpha():
        return "<num>" if any(c.isdigit() for c in word) else "<pun>"
    c = canonical(word)
    key = c[0] + "".join(ch for ch in c[1:] if ch not in VOWELS)
    if len(key) < 2:
        return c                              # short function words keep their identity
    return key[:8]


def skeleton_id(word, buckets):
    return 1 + (zlib.crc32(skeleton(word).encode("utf8")) % (buckets - 1))   # 0 = special/pad


N_WORD_FEATS = 6


def word_feats(raw):
    # cheap surface signals the canonical form intentionally throws away
    return [
        1.0 if is_emoji(raw) else 0.0,
        1.0 if is_punct(raw) else 0.0,
        1.0 if any(c.isdigit() for c in raw) else 0.0,
        1.0 if ELONG_RE.search(raw) else 0.0,             # "bohoooot", "😂😂😂😂"
        1.0 if raw.isupper() and len(raw) > 1 else 0.0,   # shouting
        min(len(raw), 12) / 12.0,
    ]


# ---- mojibake repair -------------------------------------------------
MOJI_MARKERS = ("Ã", "â€", "ð", "Å", "Â", "ï¿")


def fix_mojibake(s):
    # cp1252 first: that is how the corrupted file was actually written
    # (2431 of 2642 damaged tokens recover; latin-1 catches another 24)
    if not any(m in s for m in MOJI_MARKERS):
        return s
    for enc in ("cp1252", "latin-1"):
        try:
            return s.encode(enc).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return s


# ---- URL / mention collapse -----------------------------------------
URL_TAIL = {"//", "/", ":", ".", "t", "co", "www", "com", "bit", "ly", "html"}


def _url_fragment(t):
    # a URL continues through punctuation, very short tokens, known tail words and
    # anything containing a digit -- but stops at a real word so we never eat content
    lt = t.lower()
    return lt in URL_TAIL or len(lt) <= 2 or not lt.isalpha() or any(c.isdigit() for c in lt)


def collapse_noise(tokens, tags=None):
    # keeps tags aligned; a collapsed placeholder inherits tag "O" (ignored downstream)
    out_t, out_g, i = [], [], 0
    while i < len(tokens):
        w = tokens[i]
        lw = w.lower()
        if lw.startswith("http"):
            j = i + 1
            while j < len(tokens) and j - i <= 8 and _url_fragment(tokens[j]):
                j += 1
            out_t.append("<url>"); out_g.append("O"); i = j
            continue
        if w == "@" and i + 1 < len(tokens):
            out_t.append("<user>"); out_g.append("O"); i += 2
            continue
        if lw == "@user":
            out_t.append("<user>"); out_g.append("O"); i += 1
            continue
        out_t.append(w)
        out_g.append(tags[i] if tags is not None else "O")
        i += 1
    return (out_t, out_g) if tags is not None else (out_t, None)


# ---- spelling-drift perturbation -------------------------------------
def _drop_a_vowel(x):
    # "karna" -> "krna": the single most common Romanized shorthand move
    return x[0] + re.sub(r"[aeiou]", "", x[1:], count=1)


DRIFT_OPS = [
    lambda x, r: x.replace("a", "aa", 1),
    lambda x, r: x.replace("i", "ee", 1),
    lambda x, r: x.replace("k", "c", 1),
    lambda x, r: x.replace("v", "w", 1),
    lambda x, r: x.replace("j", "z", 1),
    lambda x, r: x.replace("f", "ph", 1),
    lambda x, r: _drop_a_vowel(x),
    lambda x, r: x + r.choice(["", "h", "z"]),
    lambda x, r: x[:-1] if len(x) > 4 else x,
    lambda x, r: x[0] + x[1] * 2 + x[2:],
]


def drift_word(w, rng, strength):
    if len(w) < 3 or not w.isalpha() or w in PLACEHOLDERS:
        return w
    out = w
    for _ in range(strength):
        if rng.random() < 0.55:
            try:
                out = DRIFT_OPS[rng.randrange(len(DRIFT_OPS))](out, rng) or out
            except Exception:
                pass
    if strength and len(out) > 3 and rng.random() < 0.12:      # fat-finger swap
        i = rng.randrange(len(out) - 1)
        out = out[:i] + out[i + 1] + out[i] + out[i + 2:]
    return out


def drift_words(words, rng, strength):
    return [drift_word(w, rng, strength) for w in words]

In [ ]:
# sanity check on the normalization layer
demo = [
    "bhai order cancel krdo please, urgent meeting h",
    "bhaaai ordr cancell kr do plzz urgnt meting hai",
    "Bohot badhiya service 🙄🙄🙄",
    "aap busy ho? kya kar rahe ho / kya krre ho",
]
for d in demo:
    toks = segment(d)
    print(f"raw    : {d}")
    print(f"canon  : {' '.join(canonical(t) for t in toks)}")
    print(f"skel   : {' '.join(skeleton(t) for t in toks)}")
    print()

noisy = "@ AdilNisarButt pakistan ka he … https // t . co / oxf8tr3bly".split()
print("collapse:", collapse_noise(noisy)[0])
print("collapse:", collapse_noise("http bhai order kaha hai".split())[0])
print("mojibake:", fix_mojibake("ðŸ˜… â€¦"))
for grp in [["kar", "kr", "karr", "kaar"], ["nahi", "nhi", "nahee"], ["ki", "ka", "ko"]]:
    print([f"{w}->{skeleton(w)}" for w in grp])

---
## 2 · Data

Both corpora are fetched from public mirrors and cached under `out_dir/data`. Before downloading,
the loader scans `/kaggle/input` for the same filenames — so with internet off, attach the files
as a Kaggle Dataset and everything else runs unchanged.

**Sources and licensing, for the whitepaper:**

- *SentiMix Hinglish* — Patwa et al., **SemEval-2020 Task 9: Overview of Sentiment Analysis of
  Code-Mixed Tweets**, ACL Anthology `2020.semeval-1.100`; original competition CodaLab #20654.
  CodaLab requires registration and doesn't serve files to a notebook, so we pull the original
  CoNLL files from a participant mirror. Gold **test** labels were never released, so we train on
  the 15,130-row train file and split the 3,000-row labelled validation file in half for val/test.
  State this in the report — it makes our number comparable to, not identical with, the official
  leaderboard protocol.
- *L3Cube-HingLID* — Nayak & Joshi, **L3Cube-HingCorpus and HingBERT** (arXiv:2204.08398).
  CC BY-NC-SA 4.0 — non-commercial, fine for a competition submission, cite it.

One data-quality note worth a line in the report: the labelled `Train.txt` and `Validation.txt`
are clean UTF-8 (4,041 emoji tokens in train; 17.4% of tweets carry at least one), but the
unlabelled `Test.txt` was written UTF-8-as-cp1252, so every emoji arrives as `ðŸ˜…`.
`fix_mojibake` repairs 2,431 of its 2,642 damaged tokens and recovers 610 emoji tokens into the
MLM corpus. Without it those tokens would train the tokenizer on garbage byte sequences.

In [ ]:
# =====================================================================
# DOWNLOAD
# =====================================================================
SEMEVAL_BASE = ("https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/"
                "Pre%20Processing%20Code/Original%20Dataset")
HINGLID_BASE = "https://raw.githubusercontent.com/l3cube-pune/code-mixed-nlp/main/L3Cube-HingLID"

FILES = {
    "sentimix_train.txt": f"{SEMEVAL_BASE}/Train.txt",
    "sentimix_val.txt":   f"{SEMEVAL_BASE}/Validation.txt",
    "sentimix_test.txt":  f"{SEMEVAL_BASE}/Test.txt",          # unlabelled (gold never released)
    "hinglid_train.txt":  f"{HINGLID_BASE}/train.txt",
    "hinglid_val.txt":    f"{HINGLID_BASE}/validation.txt",
    "hinglid_test.txt":   f"{HINGLID_BASE}/test.txt",
}
# filenames the /kaggle/input scan will also accept for each local name
ALIASES = {
    "sentimix_train.txt": ["Train.txt"], "sentimix_val.txt": ["Validation.txt"],
    "sentimix_test.txt": ["Test.txt"],
    "hinglid_train.txt": ["train.txt"], "hinglid_val.txt": ["validation.txt"],
    "hinglid_test.txt": ["test.txt"],
}


def find_in_kaggle_input(name):
    root = "/kaggle/input"
    if not os.path.isdir(root):
        return None
    wanted = {name} | set(ALIASES.get(name, []))
    for dirpath, _, files in os.walk(root):
        for fn in files:
            if fn in wanted:
                return os.path.join(dirpath, fn)
    return None


def fetch(name, url, tries=3):
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest, "cached"
    local = find_in_kaggle_input(name)
    if local:
        shutil.copyfile(local, dest)
        return dest, "kaggle-input"
    last = None
    for attempt in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=90) as r, open(dest + ".part", "wb") as f:
                shutil.copyfileobj(r, f)
            os.replace(dest + ".part", dest)
            return dest, "downloaded"
        except Exception as e:
            last = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(
        f"Could not obtain {name}.\n"
        f"  url  : {url}\n"
        f"  error: {type(last).__name__}: {last}\n"
        f"  fix  : turn Kaggle Internet ON (Settings -> Internet), or attach the file as a "
        f"Kaggle Dataset (accepted names: {[name] + ALIASES.get(name, [])})")


paths = {}
for name, url in FILES.items():
    p, how = fetch(name, url)
    paths[name] = p
    print(f"  {name:<22} {os.path.getsize(p)/1e6:>7.2f} MB  [{how}]")

In [ ]:
# =====================================================================
# PARSERS
# =====================================================================
SENT_LABELS = ["negative", "neutral", "positive"]
SENT2ID = {k: i for i, k in enumerate(SENT_LABELS)}
LID_LABELS = ["HI", "EN"]
LID2ID = {k: i for i, k in enumerate(LID_LABELS)}
SEMEVAL_LID_MAP = {"Hin": "HI", "Eng": "EN"}     # SentiMix tagset -> ours; rest ignored


def parse_sentimix(path):
    # CoNLL: "meta \t uid \t sentiment", then "token \t langtag" lines, blank line between tweets
    rows, toks, lids, meta = [], [], [], None

    def flush():
        if meta and toks:
            t, g = collapse_noise(toks, lids) if cfg.collapse_urls_mentions else (toks, lids)
            rows.append({"uid": meta[0], "label": meta[1], "words": t, "lid": g})

    with open(path, encoding="utf8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\r\n")
            if line.startswith("meta\t"):
                flush()
                p = line.split("\t")
                meta = (p[1], p[2].strip() if len(p) > 2 and p[2].strip() else None)
                toks, lids = [], []
                continue
            if not line.strip():
                flush(); meta, toks, lids = None, [], []
                continue
            p = line.split("\t")
            if not p[0]:
                continue
            toks.append(fix_mojibake(p[0]))
            raw_tag = p[1].strip() if len(p) > 1 else ""
            lids.append(SEMEVAL_LID_MAP.get(raw_tag, "O"))
    flush()
    return [r for r in rows if r["label"] in SENT2ID or r["label"] is None]


def parse_hinglid(path):
    rows, cur_w, cur_t = [], [], []
    with open(path, encoding="utf8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\r\n")
            if not line.strip():
                if cur_w:
                    rows.append({"words": cur_w, "lid": cur_t})
                cur_w, cur_t = [], []
                continue
            p = line.split("\t")
            if len(p) >= 2 and p[0]:
                cur_w.append(fix_mojibake(p[0]))
                cur_t.append(p[1].strip() if p[1].strip() in LID2ID else "O")
    if cur_w:
        rows.append({"words": cur_w, "lid": cur_t})
    if cfg.collapse_urls_mentions:
        for r in rows:
            r["words"], r["lid"] = collapse_noise(r["words"], r["lid"])
    return rows


sm_train = parse_sentimix(paths["sentimix_train.txt"])
sm_val_all = parse_sentimix(paths["sentimix_val.txt"])
sm_unlabelled = parse_sentimix(paths["sentimix_test.txt"])

# gold test labels were never released -> split the labelled validation file 50/50
random.Random(cfg.seed).shuffle(sm_val_all)
half = len(sm_val_all) // 2
sm_val, sm_test = sm_val_all[:half], sm_val_all[half:]

lid_train = parse_hinglid(paths["hinglid_train.txt"])
lid_val = parse_hinglid(paths["hinglid_val.txt"])
lid_test = parse_hinglid(paths["hinglid_test.txt"])

print(f"SentiMix Hinglish : train {len(sm_train):>6}  val {len(sm_val):>5}  test {len(sm_test):>5}"
      f"  (+{len(sm_unlabelled)} unlabelled)")
print(f"  labels          : {Counter(r['label'] for r in sm_train).most_common()}")
print(f"  own LID tags    : {Counter(t for r in sm_train for t in r['lid']).most_common()}")
print(f"HingLID           : train {len(lid_train):>6}  val {len(lid_val):>5}  test {len(lid_test):>5}")
print(f"  tags            : {Counter(t for r in lid_train for t in r['lid']).most_common()}")

n_emo = sum(1 for r in sm_train if any(is_emoji(w) for w in r["words"]))
print(f"\nSentiMix train tweets carrying at least one emoji: {n_emo} ({100*n_emo/len(sm_train):.1f}%)")
assert all(len(r["words"]) == len(r["lid"]) for r in sm_train + lid_train), "tag misalignment"
for r in sm_train[:3]:
    print(f"  [{r['label']:<8}] {' '.join(r['words'])[:110]}")

In [ ]:
# =====================================================================
# MLM CORPUS  -- every Hinglish sentence we have, labelled or not
# =====================================================================
mlm_corpus = []
for group in (sm_train, sm_val, sm_test, sm_unlabelled, lid_train, lid_val, lid_test):
    mlm_corpus += [r["words"] for r in group]
random.Random(cfg.seed).shuffle(mlm_corpus)

n_tok = sum(len(w) for w in mlm_corpus)
print(f"MLM corpus: {len(mlm_corpus):,} sentences / {n_tok:,} tokens "
      f"({n_tok/len(mlm_corpus):.1f} tokens per sentence)")
print(f"unique surface words     : {len({w for s in mlm_corpus for w in s}):,}")
print(f"unique canonical forms   : {len({canonical(w) for s in mlm_corpus for w in s}):,}")
print(f"unique skeletons         : {len({skeleton(w) for s in mlm_corpus for w in s}):,}")
print("\nThe three numbers above are the tokenization argument in miniature: canonicalization")
print("collapses the surface space, and the skeleton channel collapses it much further again.")

---
## 3 · Tokenizer

Byte-level BPE trained on the **canonicalized** corpus. This is the first half of the
"justify your tokenization choices" deliverable.

- Byte-level alphabet ⇒ **no OOV token, ever**. Romanized words never shatter into `[UNK]`, which
  is exactly the failure mode the PS describes for native-script Indic models.
- Trained on canonical forms, so `karna` / `karnaa` / `kaarna` share one merge path instead of
  burning three vocabulary slots.
- Every emoji in the corpus becomes an **atomic token**, so a pragmatic marker is one embedding
  lookup rather than 3–4 byte fragments.

**Fertility** (subwords per word), printed below, is the number to put in the whitepaper: it
directly measures fragmentation, and it's the axis on which a native-script vocabulary fails on
Romanized input. The cell also prints fertility for the drifted variant of each word — the gap
between the two is the tokenizer's own contribution to spelling-drift robustness, before the model
sees anything.

In [ ]:
# =====================================================================
# TOKENIZER
# =====================================================================
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SPECIALS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

tok = Tokenizer(models.BPE(unk_token="[UNK]"))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=cfg.vocab_size,
    special_tokens=SPECIALS,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True,
)


def canon_corpus_iter():
    for words in mlm_corpus:
        yield " ".join(canonical(w) for w in words)


tok.train_from_iterator(canon_corpus_iter(), trainer=trainer, length=len(mlm_corpus))

emoji_vocab = sorted({w for s in mlm_corpus for w in s if is_emoji(w)})
n_added = tok.add_tokens(emoji_vocab + list(PLACEHOLDERS))

PAD_ID = tok.token_to_id("[PAD]")
UNK_ID = tok.token_to_id("[UNK]")
CLS_ID = tok.token_to_id("[CLS]")
SEP_ID = tok.token_to_id("[SEP]")
MASK_ID = tok.token_to_id("[MASK]")
VOCAB = tok.get_vocab_size(with_added_tokens=True)
tok.save(os.path.join(cfg.out_dir, "tokenizer.json"))

sample = mlm_corpus[:4000]
n_words = sum(len(s) for s in sample)
n_subs = sum(len(tok.encode(canonical(w)).ids) for s in sample for w in s)
_rng = random.Random(0)
n_subs_drift = sum(len(tok.encode(canonical(drift_word(w, _rng, 2))).ids)
                   for s in sample for w in s)
print(f"vocab size              : {VOCAB}  (+{n_added} emoji/placeholder tokens)")
print(f"fertility (clean)       : {n_subs/n_words:.3f} subwords per word")
print(f"fertility (drift lvl 2) : {n_subs_drift/n_words:.3f} "
      f"(+{100*(n_subs_drift-n_subs)/n_subs:.1f}%)")
for w in ["krdo", "kardo", "nahiiii", "bohot", "bohoooot", "chahiye"]:
    print(f"  {w:<10} -> {tok.encode(canonical(w)).tokens}")

In [ ]:
# =====================================================================
# FEATURIZER  -- words -> (subword ids, skeleton ids, surface feats, word spans)
# =====================================================================
ZERO_FEAT = [0.0] * N_WORD_FEATS


class Featurizer:
    # word-level cache: the unique-word set is ~110k across both corpora
    def __init__(self, tokenizer, cfg):
        self.tok, self.cfg, self._cache = tokenizer, cfg, {}

    def word(self, raw):
        hit = self._cache.get(raw)
        if hit is not None:
            return hit
        ids = (self.tok.encode(canonical(raw)).ids or [UNK_ID])[:8]
        out = (ids, skeleton_id(raw, self.cfg.skeleton_buckets), word_feats(raw))
        self._cache[raw] = out
        return out

    def encode(self, words, max_len=None):
        max_len = max_len or self.cfg.max_len
        ids, skel, feats, spans = [CLS_ID], [0], [ZERO_FEAT], []
        for w in words:
            sub, sk, ft = self.word(w)
            if len(ids) + len(sub) + 1 > max_len:
                break
            spans.append((len(ids), len(ids) + len(sub)))
            ids += sub; skel += [sk] * len(sub); feats += [ft] * len(sub)
        ids.append(SEP_ID); skel.append(0); feats.append(ZERO_FEAT)
        return {"input_ids": ids, "skel_ids": skel, "feats": feats, "word_spans": spans}


feat = Featurizer(tok, cfg)
trunc = sum(1 for r in sm_train[:3000] if len(feat.encode(r["words"])["word_spans"]) < len(r["words"]))
print(f"truncated at max_len={cfg.max_len}: {trunc}/3000 SentiMix rows")


def pad_batch(encoded, pad_to=None):
    L = pad_to or max(len(e["input_ids"]) for e in encoded)
    B = len(encoded)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    skel = torch.zeros((B, L), dtype=torch.long)
    fts = torch.zeros((B, L, N_WORD_FEATS), dtype=torch.float)
    att = torch.zeros((B, L), dtype=torch.bool)
    for i, e in enumerate(encoded):
        n = min(len(e["input_ids"]), L)
        ids[i, :n] = torch.tensor(e["input_ids"][:n], dtype=torch.long)
        skel[i, :n] = torch.tensor(e["skel_ids"][:n], dtype=torch.long)
        fts[i, :n] = torch.tensor(e["feats"][:n], dtype=torch.float)
        att[i, :n] = True
    return {"input_ids": ids, "skel_ids": skel, "feats": fts, "attention_mask": att}

---
## 4 · Model

A pre-LN transformer encoder written from scratch — no pretrained checkpoint, nothing to strip out.
Second half of the "justify your architecture choices" deliverable.

**Input embedding = token + position + skeleton + surface-feature projection.**

- The **skeleton** term is the robustness mechanism. Two spellings may take different BPE paths but
  share a skeleton bucket, so gradient signal pools across drift variants. During MLM the skeleton
  at masked positions is zeroed, otherwise it leaks the answer.
- The **surface-feature** term carries the six signals canonicalization deliberately destroys
  (is-emoji, is-punct, has-digit, elongation, all-caps, length). Elongation and caps are exactly
  the intensity markers the PS's pragmatics bullet is about.

Attention uses `F.scaled_dot_product_attention`, which dispatches to the fused/flash kernel where
available — that's where the single-digit-millisecond target comes from. Two heads share the
encoder: sentence sentiment over pooled `[CLS]`, and a token-level LID tagger.

In [ ]:
# =====================================================================
# MODEL
# =====================================================================
class Block(nn.Module):
    def __init__(self, d, h, ff, p):
        super().__init__()
        assert d % h == 0
        self.h, self.dh = h, d // h
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.drop = nn.Dropout(p)

    def forward(self, x, keep_mask):
        B, L, D = x.shape
        h = self.ln1(x)
        q, k, v = self.qkv(h).split(D, dim=-1)
        q = q.view(B, L, self.h, self.dh).transpose(1, 2)
        k = k.view(B, L, self.h, self.dh).transpose(1, 2)
        v = v.view(B, L, self.h, self.dh).transpose(1, 2)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=keep_mask)
        o = o.transpose(1, 2).reshape(B, L, D)
        x = x + self.drop(self.proj(o))
        x = x + self.drop(self.mlp(self.ln2(x)))
        return x


class PolyglotEncoder(nn.Module):
    def __init__(self, cfg, vocab):
        super().__init__()
        d = cfg.d_model
        self.tok_emb = nn.Embedding(vocab, d, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(cfg.max_len, d)
        self.skel_emb = nn.Embedding(cfg.skeleton_buckets, d, padding_idx=0)
        self.feat_proj = nn.Linear(N_WORD_FEATS, d, bias=False)
        self.ln_in = nn.LayerNorm(d)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(
            [Block(d, cfg.n_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.n_layers)])
        self.ln_out = nn.LayerNorm(d)
        self.use_skel = True        # flipped to False for the ablation in §10
        self.use_feats = True
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, attention_mask, skel_ids=None, feats=None):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        if skel_ids is not None and self.use_skel:
            x = x + self.skel_emb(skel_ids)
        if feats is not None and self.use_feats:
            x = x + self.feat_proj(feats)
        x = self.drop(self.ln_in(x))
        # bool mask: True = this key participates. [CLS] is always present, so no query
        # row is ever fully masked (which would produce NaNs).
        keep = attention_mask[:, None, None, :].bool()
        for blk in self.blocks:
            x = blk(x, keep)
        return self.ln_out(x)


class PolyglotForTasks(nn.Module):
    def __init__(self, cfg, vocab):
        super().__init__()
        d = cfg.d_model
        self.encoder = PolyglotEncoder(cfg, vocab)
        self.mlm_head = nn.Linear(d, vocab, bias=False)
        self.mlm_head.weight = self.encoder.tok_emb.weight          # tied
        self.mlm_bias = nn.Parameter(torch.zeros(vocab))
        self.pool = nn.Sequential(nn.Linear(d, d), nn.Tanh())
        self.sent_head = nn.Linear(d, len(SENT_LABELS))
        self.lid_head = nn.Linear(d, len(LID_LABELS))

    def forward(self, b, task):
        h = self.encoder(b["input_ids"], b["attention_mask"], b["skel_ids"], b["feats"])
        if task == "mlm":
            return self.mlm_head(h) + self.mlm_bias
        if task == "lid":
            return self.lid_head(h)
        return self.sent_head(self.pool(h[:, 0]))


model = PolyglotForTasks(cfg, VOCAB).to(DEVICE)
print("heads: sent, lid, mlm | device:", DEVICE)

In [ ]:
# =====================================================================
# FOOTPRINT REPORT  -- PS constraint: max 500M parameters
# =====================================================================
def count_params(m):
    seen, total = set(), 0
    for p in m.parameters():
        if id(p) in seen:              # tied weights counted once
            continue
        seen.add(id(p)); total += p.numel()
    return total


total = count_params(model)
budget = 500_000_000
footprint = {
    "total_params": total,
    "encoder_params": count_params(model.encoder),
    "embedding_params": count_params(model.encoder.tok_emb) + count_params(model.encoder.skel_emb),
    "fp32_mb": total * 4 / 1e6,
    "int8_mb": total / 1e6,
    "budget": budget,
    "budget_used_pct": 100 * total / budget,
}
for k, v in footprint.items():
    print(f"{k:>18}: {v:,.3f}" if isinstance(v, float) else f"{k:>18}: {v:,}")
assert total < budget, "over the 500M parameter budget"

---
## 5 · MLM pretraining

Standard 15% masking (80/10/10 mask/random/keep) over every sentence from both corpora. The
**skeleton channel is zeroed at masked positions** so the model can't cheat, and masking happens on
the canonicalized stream so the objective is to reconstruct drift-normalized word identity rather
than one particular spelling.

Be realistic about scale: this is ~1.7M tokens, three orders of magnitude below what HingBERT saw.
This encoder will not match a HingRoBERTa fine-tune and the report should say so plainly. §10 lists
the two ways to close the gap.

In [ ]:
# =====================================================================
# MLM PRETRAINING
# =====================================================================
class MLMDataset(Dataset):
    def __init__(self, corpus, feat):
        self.corpus, self.feat = corpus, feat

    def __len__(self):
        return len(self.corpus)

    def __getitem__(self, i):
        return self.feat.encode(self.corpus[i])


def mlm_collate(batch):
    b = pad_batch(batch)
    ids = b["input_ids"]
    labels = ids.clone()

    special = (ids == PAD_ID) | (ids == CLS_ID) | (ids == SEP_ID)
    prob = torch.full(ids.shape, cfg.mask_prob).masked_fill(special, 0.0)
    picked = torch.bernoulli(prob).bool()
    labels[~picked] = -100

    r = torch.rand(ids.shape)
    ids[picked & (r < 0.8)] = MASK_ID
    rand_pos = picked & (r >= 0.8) & (r < 0.9)
    ids[rand_pos] = torch.randint(len(SPECIALS), VOCAB, (int(rand_pos.sum()),))

    b["input_ids"] = ids
    b["skel_ids"] = b["skel_ids"].masked_fill(picked, 0)     # no leakage
    b["labels"] = labels
    return b


def to_dev(b):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}


mlm_dl = DataLoader(MLMDataset(mlm_corpus, feat), batch_size=cfg.mlm_bs, shuffle=True,
                    collate_fn=mlm_collate, num_workers=0, drop_last=True)

opt = torch.optim.AdamW(model.parameters(), lr=cfg.mlm_lr, weight_decay=0.01, betas=(0.9, 0.98))
steps = max(1, cfg.mlm_epochs * len(mlm_dl))
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.mlm_lr, total_steps=steps, pct_start=0.1)
scaler = make_scaler(cfg.amp and DEVICE == "cuda")

model.train()
mlm_history, t0 = [], time.time()
for ep in range(cfg.mlm_epochs):
    running, seen = 0.0, 0
    for batch in mlm_dl:
        batch = to_dev(batch)
        opt.zero_grad(set_to_none=True)
        with autocast_ctx(cfg.amp and DEVICE == "cuda"):
            logits = model(batch, "mlm")
            loss = F.cross_entropy(logits.view(-1, VOCAB), batch["labels"].view(-1),
                                   ignore_index=-100)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        running += loss.item(); seen += 1
    avg = running / seen
    mlm_history.append({"epoch": ep, "loss": avg, "ppl": math.exp(min(20, avg))})
    print(f"  epoch {ep:>2}  mlm loss {avg:.4f}  ppl {math.exp(min(20, avg)):.1f}"
          f"  ({time.time()-t0:.0f}s)")
torch.save(model.state_dict(), os.path.join(cfg.out_dir, "pretrained.pt"))
print(f"MLM pretraining done in {time.time()-t0:.1f}s")

---
## 6 · Multi-task fine-tuning

One encoder, two heads, interleaved batches:

| Task | Data | Head |
|---|---|---|
| `sent` | SentiMix Hinglish | 3-way over pooled `[CLS]` |
| `lid` | HingLID **+ SentiMix's own word tags** | token-level, first-subword labelling |

Folding SentiMix's language tags into the LID task matters: it puts token-level code-switching
supervision inside the *same domain* as the sentiment data, rather than only on scraped tweets.

`augment_drift_p` re-spells 30% of training rows every epoch using the §1 perturbation operators.
This is augmentation on real data — no synthetic sentences — and it's the cleanest single ablation
for the writeup: set it to 0.0, retrain, compare the §7 drift curves.

In [ ]:
# =====================================================================
# TASK DATASETS
# =====================================================================
class SentDataset(Dataset):
    def __init__(self, rows, feat, augment=0.0):
        self.rows, self.feat = rows, feat
        self.augment, self.rng = augment, random.Random(cfg.seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        words = r["words"]
        if self.augment and self.rng.random() < self.augment:
            words = drift_words(words, self.rng, cfg.augment_drift_strength)
        e = self.feat.encode(words)
        e["label"] = SENT2ID[r["label"]]
        return e


class LidDataset(Dataset):
    # word-level tags projected onto the FIRST subword of each word; the rest are ignored
    def __init__(self, rows, feat, augment=0.0):
        self.rows = [r for r in rows if any(t in LID2ID for t in r["lid"])]
        self.feat = feat
        self.augment, self.rng = augment, random.Random(cfg.seed + 1)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        words, tags = r["words"], r["lid"]
        if self.augment and self.rng.random() < self.augment:
            words = drift_words(words, self.rng, cfg.augment_drift_strength)
        e = self.feat.encode(words)
        lab = [-100] * len(e["input_ids"])
        for wi, (s, _) in enumerate(e["word_spans"]):
            t = tags[wi] if wi < len(tags) else "O"
            if t in LID2ID:
                lab[s] = LID2ID[t]
        e["token_labels"] = lab
        return e


def sent_collate(batch):
    b = pad_batch(batch)
    b["label"] = torch.tensor([e["label"] for e in batch], dtype=torch.long)
    return b


def lid_collate(batch):
    b = pad_batch(batch)
    L = b["input_ids"].shape[1]
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, e in enumerate(batch):
        n = min(len(e["token_labels"]), L)
        lab[i, :n] = torch.tensor(e["token_labels"][:n], dtype=torch.long)
    b["token_labels"] = lab
    return b


AUG = cfg.augment_drift_p
COLLATE = {"sent": sent_collate, "lid": lid_collate}
SPLITS = {
    "sent": (SentDataset(sm_train, feat, AUG), SentDataset(sm_val, feat), SentDataset(sm_test, feat)),
    "lid": (LidDataset(lid_train + sm_train, feat, AUG), LidDataset(lid_val, feat),
            LidDataset(lid_test, feat)),
}

ACTIVE = [t for t, on in [("sent", cfg.use_sent), ("lid", cfg.use_lid)] if on]
TASK_W = dict(cfg.task_weights)

train_dls, val_dls, test_dls = {}, {}, {}
for name in ACTIVE:
    tr, va, te = SPLITS[name]
    train_dls[name] = DataLoader(tr, batch_size=cfg.ft_bs, shuffle=True,
                                 collate_fn=COLLATE[name], num_workers=0, drop_last=True)
    val_dls[name] = DataLoader(va, batch_size=128, shuffle=False,
                               collate_fn=COLLATE[name], num_workers=0)
    test_dls[name] = DataLoader(te, batch_size=128, shuffle=False,
                                collate_fn=COLLATE[name], num_workers=0)
    print(f"{name:>5}: train {len(tr):>6}  val {len(va):>5}  test {len(te):>5}  "
          f"batches/epoch {len(train_dls[name])}")

In [ ]:
# =====================================================================
# MULTI-TASK TRAINING LOOP
# =====================================================================
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix


def task_loss(model, batch, task):
    if task == "lid":
        logits = model(batch, "lid")
        return F.cross_entropy(logits.reshape(-1, len(LID_LABELS)),
                               batch["token_labels"].reshape(-1), ignore_index=-100)
    return F.cross_entropy(model(batch, "sent"), batch["label"], label_smoothing=0.05)


@torch.no_grad()
def predict_task(model, dl, task):
    model.eval()
    P, Y = [], []
    for b in dl:
        b = to_dev(b)
        if task == "lid":
            lg = model(b, "lid").argmax(-1)
            m = b["token_labels"] != -100
            P.append(lg[m].cpu()); Y.append(b["token_labels"][m].cpu())
        else:
            P.append(model(b, "sent").argmax(-1).cpu()); Y.append(b["label"].cpu())
    model.train()
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def evaluate(model, loaders):
    out = {}
    for task, dl in loaders.items():
        P, Y = predict_task(model, dl, task)
        out[f"{task}_acc"] = accuracy_score(Y, P)
        out[f"{task}_macro_f1"] = f1_score(Y, P, average="macro")
        out[f"{task}_weighted_f1"] = f1_score(Y, P, average="weighted")
    return out


opt = torch.optim.AdamW(model.parameters(), lr=cfg.ft_lr, weight_decay=0.01)
plan = [t for t, dl in train_dls.items() for _ in range(len(dl))]
steps = max(1, cfg.ft_epochs * len(plan))
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.ft_lr, total_steps=steps, pct_start=0.1)
scaler = make_scaler(cfg.amp and DEVICE == "cuda")

history, best, t0 = [], -1.0, time.time()
for ep in range(cfg.ft_epochs):
    order = plan[:]
    random.shuffle(order)
    iters = {k: iter(v) for k, v in train_dls.items()}
    agg, cnt = defaultdict(float), defaultdict(int)

    for task in order:
        try:
            batch = next(iters[task])
        except StopIteration:
            iters[task] = iter(train_dls[task]); batch = next(iters[task])
        batch = to_dev(batch)
        opt.zero_grad(set_to_none=True)
        with autocast_ctx(cfg.amp and DEVICE == "cuda"):
            loss = TASK_W.get(task, 1.0) * task_loss(model, batch, task)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        agg[task] += loss.item(); cnt[task] += 1

    m = evaluate(model, val_dls)
    m["epoch"] = ep
    m["train_loss"] = {k: agg[k] / max(cnt[k], 1) for k in agg}
    history.append(m)
    score = m.get("sent_weighted_f1", m.get("lid_weighted_f1", 0))    # select on the headline task
    print(f"[epoch {ep}] " + "  ".join(
        f"{k}={v:.4f}" for k, v in m.items() if isinstance(v, float)))
    if score > best:
        best = score
        torch.save(model.state_dict(), os.path.join(cfg.out_dir, "best.pt"))

model.load_state_dict(torch.load(os.path.join(cfg.out_dir, "best.pt"), map_location=DEVICE))
print(f"fine-tuning done in {time.time()-t0:.1f}s  (best val weighted F1 {best:.4f})")

---
## 7 · Evaluation, robustness and error analysis

Four blocks, mapping onto the PS deliverables:

1. **Clean test metrics + leaderboard context.** SemEval-2020 Task 9 Hinglish reported a best
   system at **75.0 weighted F1**, with BERT-family models and ensembles at the top. Our protocol
   differs — gold test labels were never released, so we score on half the labelled validation
   file. Say that plainly rather than claiming a leaderboard position.
2. **Spelling-drift perturbation sweep** — the *same real test sentences*, re-spelled at levels
   0–3. Labels are invariant by construction, so any drop is pure brittleness. This is the headline
   robustness figure and it answers the PS's robustness constraint directly.
3. **Emoji ablation** — strip every emoji and re-score, reported separately on the ~17% of tweets
   that carry one. If that subset barely moves, the "emojis as semantic modulators" claim is
   unearned and the report should admit it. Note the subset is small (~250 rows in test), so treat
   the delta as indicative, not precise.
4. **Sliced error analysis** — the explicit PS deliverable. Accuracy broken out by code-mix ratio
   (from gold language tags), sentence length, emoji presence, elongation, and tokenizer fertility,
   plus a confusion matrix, the most confident mistakes, and LID accuracy on *ambiguous* words
   (words that appear as both Hindi and English in training). That last one is the real test of
   whether the encoder uses context rather than word identity.

In [ ]:
# =====================================================================
# CLEAN TEST METRICS
# =====================================================================
clean_metrics = evaluate(model, test_dls)
print("TEST (clean):")
for k in sorted(clean_metrics):
    print(f"  {k:>20}: {clean_metrics[k]:.4f}")

LABEL_NAMES = {"sent": SENT_LABELS, "lid": LID_LABELS}
for task in ACTIVE:
    P, Y = predict_task(model, test_dls[task], task)
    names = LABEL_NAMES[task]
    print(f"\n--- {task} ---")
    print(classification_report(Y, P, labels=list(range(len(names))), target_names=names,
                                digits=3, zero_division=0))

SEMEVAL_REFERENCE = {
    "SemEval-2020 Task 9 best system (weighted F1)": 0.750,
    "this model (weighted F1)": clean_metrics.get("sent_weighted_f1", float("nan")),
}
print("Hinglish sentiment in context:")
for k, v in SEMEVAL_REFERENCE.items():
    print(f"  {k:<48} {v:.3f}")
print("  (official test labels unreleased -> we score on half the labelled val file)")

In [ ]:
# =====================================================================
# ROBUSTNESS: SPELLING-DRIFT SWEEP  +  EMOJI ABLATION
# =====================================================================
import copy


def perturb(rows, strength, seed=123):
    # re-spells the SAME real sentences; emojis, punctuation and placeholders untouched
    rng = random.Random(seed)
    out = []
    for r in rows:
        nr = dict(r)
        nr["words"] = drift_words(r["words"], rng, strength)
        out.append(nr)
    return out


def strip_emojis(rows):
    out = []
    for r in rows:
        nr = dict(r)
        keep = [i for i, w in enumerate(r["words"]) if not is_emoji(w)]
        nr["words"] = [r["words"][i] for i in keep] or ["."]
        if "lid" in r:
            nr["lid"] = [r["lid"][i] for i in keep] or ["O"]
        out.append(nr)
    return out


def loader_for(task, rows):
    ds = SentDataset(rows, feat) if task == "sent" else LidDataset(rows, feat)
    return DataLoader(ds, batch_size=128, shuffle=False, collate_fn=COLLATE[task], num_workers=0)


TEST_ROWS = {"sent": sm_test, "lid": lid_test}

drift_results = []
for lvl in [0, 1, 2, 3]:
    row = {"drift": lvl}
    for task in ACTIVE:
        P, Y = predict_task(model, loader_for(task, perturb(TEST_ROWS[task], lvl)), task)
        row[f"{task}_weighted_f1"] = f1_score(Y, P, average="weighted")
    drift_results.append(row)
    print(f"drift {lvl}:  " + "  ".join(f"{t}={row[f'{t}_weighted_f1']:.4f}" for t in ACTIVE))

head = ACTIVE[0]
base = drift_results[0][f"{head}_weighted_f1"] or 1e-9
worst = drift_results[-1][f"{head}_weighted_f1"]
print(f"\n{head} weighted-F1 retention at max perturbation: {100*worst/base:.1f}%")

# ---- emoji ablation, on the real SentiMix test set --------------------
emoji_ablation = {}
if "sent" in ACTIVE:
    with_emo = [r for r in sm_test if any(is_emoji(w) for w in r["words"])]
    without_emo = [r for r in sm_test if not any(is_emoji(w) for w in r["words"])]

    def wf1(rows):
        if not rows:
            return float("nan")
        P, Y = predict_task(model, loader_for("sent", rows), "sent")
        return f1_score(Y, P, average="weighted")

    emoji_ablation = {
        "full_test": wf1(sm_test),
        "full_test_stripped": wf1(strip_emojis(sm_test)),
        "emoji_subset": wf1(with_emo),
        "emoji_subset_stripped": wf1(strip_emojis(with_emo)),
        "no_emoji_subset": wf1(without_emo),
        "emoji_subset_size": len(with_emo),
    }
    print("\nEMOJI ABLATION (SentiMix test, weighted F1):")
    for k, v in emoji_ablation.items():
        print(f"  {k:>24}: {v:.4f}" if isinstance(v, float) else f"  {k:>24}: {v}")
    d = emoji_ablation["emoji_subset"] - emoji_ablation["emoji_subset_stripped"]
    print(f"  -> removing emojis costs {100*d:.2f} F1 points on the {len(with_emo)} tweets "
          f"that carry one")

In [ ]:
# =====================================================================
# ERROR ANALYSIS  (explicit PS deliverable)
# =====================================================================
@torch.no_grad()
def sent_probs(rows):
    dl = DataLoader(SentDataset(rows, feat), batch_size=128, shuffle=False,
                    collate_fn=sent_collate, num_workers=0)
    model.eval()
    out = [model(to_dev(b), "sent").softmax(-1).cpu() for b in dl]
    model.train()
    return torch.cat(out).numpy()


probs = sent_probs(sm_test)
pred = probs.argmax(1)
gold = np.array([SENT2ID[r["label"]] for r in sm_test])
conf = probs.max(1)


def code_mix_ratio(r):
    tags = [t for t in r["lid"] if t in LID2ID]
    return (sum(t == "EN" for t in tags) / len(tags)) if tags else None


def fertility(r):
    n = sum(len(feat.word(w)[0]) for w in r["words"])
    return n / max(len(r["words"]), 1)


cmr = [code_mix_ratio(r) for r in sm_test]
lens = [len(r["words"]) for r in sm_test]
fert = [fertility(r) for r in sm_test]

SLICES = [
    ("mostly Hindi (EN<20%)",   lambda i: cmr[i] is not None and cmr[i] < 0.20),
    ("balanced mix (20-60%)",   lambda i: cmr[i] is not None and 0.20 <= cmr[i] <= 0.60),
    ("mostly English (>60%)",   lambda i: cmr[i] is not None and cmr[i] > 0.60),
    ("short (<15 words)",       lambda i: lens[i] < 15),
    ("medium (15-30)",          lambda i: 15 <= lens[i] <= 30),
    ("long (>30)",              lambda i: lens[i] > 30),
    ("has emoji",               lambda i: any(is_emoji(w) for w in sm_test[i]["words"])),
    ("has elongation",          lambda i: any(ELONG_RE.search(w) for w in sm_test[i]["words"])),
    ("low fertility (<1.5)",    lambda i: fert[i] < 1.5),
    ("high fertility (>=2.0)",  lambda i: fert[i] >= 2.0),
]

slice_report = []
print(f"{'slice':<26} {'n':>6} {'acc':>8} {'wF1':>8}")
print("-" * 52)
for name, fn in SLICES:
    idx = [i for i in range(len(sm_test)) if fn(i)]
    if len(idx) < 20:
        continue
    a = accuracy_score(gold[idx], pred[idx])
    f = f1_score(gold[idx], pred[idx], average="weighted")
    slice_report.append({"slice": name, "n": len(idx), "acc": a, "weighted_f1": f})
    print(f"{name:<26} {len(idx):>6} {a:>8.4f} {f:>8.4f}")
print("-" * 52)
print(f"{'ALL':<26} {len(sm_test):>6} {accuracy_score(gold, pred):>8.4f} "
      f"{f1_score(gold, pred, average='weighted'):>8.4f}")

print("\nconfusion matrix (rows = gold, cols = predicted):")
cm = confusion_matrix(gold, pred, labels=list(range(len(SENT_LABELS))))
print(f"{'':>10}" + "".join(f"{n:>10}" for n in SENT_LABELS))
for i, n in enumerate(SENT_LABELS):
    print(f"{n:>10}" + "".join(f"{v:>10}" for v in cm[i]))

wrong = [i for i in range(len(sm_test)) if pred[i] != gold[i]]
wrong.sort(key=lambda i: -conf[i])
print(f"\nmost confident mistakes ({len(wrong)} errors total):")
for i in wrong[:8]:
    print(f"  gold={SENT_LABELS[gold[i]]:<8} pred={SENT_LABELS[pred[i]]:<8} p={conf[i]:.2f} "
          f"| {' '.join(sm_test[i]['words'])[:95]}")

# ---- LID: accuracy on words that are genuinely ambiguous -------------
lid_error = {}
if "lid" in ACTIVE:
    seen = defaultdict(set)
    for r in lid_train + sm_train:
        for w, t in zip(r["words"], r["lid"]):
            if t in LID2ID:
                seen[w.lower()].add(t)
    ambiguous = {w for w, ts in seen.items() if len(ts) > 1}

    @torch.no_grad()
    def lid_word_preds(rows, limit=3000):
        ds = LidDataset(rows, feat)
        n = min(len(ds), limit)
        model.eval()
        out = []
        for i in range(0, n, 128):
            batch = [ds[j] for j in range(i, min(i + 128, n))]
            b = to_dev(lid_collate(batch))
            pr = model(b, "lid").argmax(-1).cpu()
            lb = b["token_labels"].cpu()
            for k, e in enumerate(batch):
                words = ds.rows[i + k]["words"]
                for wi, (s, _) in enumerate(e["word_spans"]):
                    if lb[k, s].item() != -100 and wi < len(words):
                        out.append((words[wi], int(lb[k, s]), int(pr[k, s])))
        model.train()
        return out

    wp = lid_word_preds(lid_test)
    amb = [(w, g, p) for w, g, p in wp if w.lower() in ambiguous]
    unamb = [(w, g, p) for w, g, p in wp if w.lower() not in ambiguous]
    unseen = [(w, g, p) for w, g, p in wp if w.lower() not in seen]
    lid_error = {
        "tokens_scored": len(wp),
        "ambiguous_words_in_train_vocab": len(ambiguous),
        "acc_all": float(np.mean([g == p for _, g, p in wp])) if wp else float("nan"),
        "acc_ambiguous": float(np.mean([g == p for _, g, p in amb])) if amb else float("nan"),
        "acc_unambiguous": float(np.mean([g == p for _, g, p in unamb])) if unamb else float("nan"),
        "acc_unseen_in_train": float(np.mean([g == p for _, g, p in unseen])) if unseen else float("nan"),
        "n_ambiguous": len(amb), "n_unseen": len(unseen),
    }
    print("\nLID error analysis:")
    for k, v in lid_error.items():
        print(f"  {k:>30}: {v:.4f}" if isinstance(v, float) else f"  {k:>30}: {v:,}")
    print("  (acc_unseen is the real generalisation number: words never seen in training)")

In [ ]:
# =====================================================================
# QUALITATIVE PROBE
# =====================================================================
def prep(text):
    w = segment(text)
    w, _ = collapse_noise(w, ["O"] * len(w))
    return w


@torch.no_grad()
def classify(texts):
    # inference entry point -- also used by the exported artifact in section 9
    model.eval()
    enc = [feat.encode(prep(t)) for t in texts]
    b = to_dev(pad_batch(enc))
    p = model(b, "sent").softmax(-1).cpu()
    model.train()
    return [{"text": t, "label": SENT_LABELS[int(p[i].argmax())],
             "confidence": float(p[i].max()),
             "scores": {n: float(p[i][j]) for j, n in enumerate(SENT_LABELS)}}
            for i, t in enumerate(texts)]


probes = [
    "bhai order cancel krdo please urgent meeting h",
    "bhaaai ordr cancell kr do plzz urgnt meting hai",
    "bohot badhiya service thi",
    "bohot badhiya service thi 🙄",
    "bohot badhiya service thi 😍",
    "item delivered bol rha h but mila hi nhi 😡",
    "kya baat hai yaar maza aa gaya",
]
for r in classify(probes):
    print(f"{r['label']:>8} ({r['confidence']:.2f})  | {r['text']}")

if "lid" in ACTIVE:
    s = prep("mujhe ye product bilkul pasand nahi aaya very poor quality")
    e = feat.encode(s)
    model.eval()
    with torch.no_grad():
        tags = model(to_dev(pad_batch([e])), "lid").argmax(-1)[0].cpu()
    model.train()
    print("\ntoken LID:")
    print("  " + "  ".join(f"{w}/{LID_LABELS[tags[st]]}" for w, (st, _) in zip(s, e["word_spans"])))

---
## 8 · Latency and throughput trade-offs

The PS constraint is **single-digit millisecond latency at batch 1 with high throughput** — two
different measurements that trade against each other. Three sweeps:

| Sweep | What it shows |
|---|---|
| batch size 1 → 256 | the latency/throughput frontier: where batching stops buying throughput |
| sequence length 16 → 96 at batch 1 | how latency scales with input length, i.e. the cost of long reviews vs short queries |
| fp32 / fp16 / CPU / **int8 with its accuracy** | the deployment trade-off: what precision actually costs you in F1 |

Timings include CUDA synchronisation and 20 warmup iterations. Preprocessing is measured
separately, because on a model this small it's a real share of wall clock — itself a finding.

In [ ]:
# =====================================================================
# BENCHMARK
# =====================================================================
class InferenceModel(nn.Module):
    # single-path serving graph: encoder + sentiment head, no task branching
    def __init__(self, base):
        super().__init__()
        self.encoder, self.pool, self.head = base.encoder, base.pool, base.sent_head

    def forward(self, input_ids, attention_mask, skel_ids, feats):
        h = self.encoder(input_ids, attention_mask, skel_ids, feats)
        return self.head(self.pool(h[:, 0]))


infer = InferenceModel(copy.deepcopy(model)).eval()     # deepcopy: never mutate trained weights
bench_rows = []
SEQ = 48


def make_inputs(bs, L, device):
    enc = [feat.encode(sm_test[i % len(sm_test)]["words"]) for i in range(bs)]
    b = pad_batch(enc, pad_to=L)
    return tuple(b[k].to(device) for k in ("input_ids", "attention_mask", "skel_ids", "feats"))


@torch.no_grad()
def bench(m, bs, L, device, iters=200, warmup=20, half=False):
    m = m.to(device)
    if half:
        m = m.half()
    ids, att, skel, fts = make_inputs(bs, L, device)
    if half:
        fts = fts.half()
    for _ in range(warmup):
        m(ids, att, skel, fts)
    if device == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        m(ids, att, skel, fts)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    if half:
        m.float()
    t = np.array(times)
    return {"batch": bs, "seq_len": L, "device": device,
            "p50_ms": float(np.percentile(t, 50)), "p95_ms": float(np.percentile(t, 95)),
            "p99_ms": float(np.percentile(t, 99)),
            "throughput_qps": bs / (float(np.percentile(t, 50)) / 1000.0)}


# ---- sweep 1: batch size --------------------------------------------
if DEVICE == "cuda":
    for bs in [1, 8, 32, 128, 256]:
        r = bench(infer, bs, SEQ, "cuda"); r["precision"] = "fp32"; bench_rows.append(r)
    for bs in [1, 128]:
        r = bench(infer, bs, SEQ, "cuda", half=True); r["precision"] = "fp16"; bench_rows.append(r)

infer_cpu = InferenceModel(copy.deepcopy(model)).float().cpu().eval()
torch.set_num_threads(min(4, os.cpu_count() or 1))
for bs in [1, 32]:
    r = bench(infer_cpu, bs, SEQ, "cpu", iters=60, warmup=10)
    r["precision"] = "fp32"; bench_rows.append(r)

# ---- sweep 2: sequence length at batch 1 ----------------------------
seq_rows = []
for L in [16, 32, 48, 64, 96]:
    dev = "cuda" if DEVICE == "cuda" else "cpu"
    r = bench(infer if dev == "cuda" else infer_cpu, 1, L, dev,
              iters=200 if dev == "cuda" else 60, warmup=20 if dev == "cuda" else 10)
    r["precision"] = "fp32"; seq_rows.append(r)
    print(f"seq_len {L:>3}: p50 {r['p50_ms']:.3f} ms   p99 {r['p99_ms']:.3f} ms")

# ---- sweep 3: int8, with the accuracy it costs ----------------------
int8_tradeoff = {}
try:
    q_model = torch.ao.quantization.quantize_dynamic(
        copy.deepcopy(infer_cpu), {nn.Linear}, dtype=torch.qint8).eval()
    r = bench(q_model, 1, SEQ, "cpu", iters=60, warmup=10)
    r["precision"] = "int8-dynamic"; bench_rows.append(r)

    @torch.no_grad()
    def eval_serving_model(m, rows, device="cpu"):
        ds = SentDataset(rows, feat)
        P, Y = [], []
        for i in range(0, len(ds), 64):
            batch = [ds[j] for j in range(i, min(i + 64, len(ds)))]
            b = sent_collate(batch)
            lg = m(b["input_ids"].to(device), b["attention_mask"].to(device),
                   b["skel_ids"].to(device), b["feats"].to(device))
            P.append(lg.argmax(-1).cpu()); Y.append(b["label"])
        return f1_score(torch.cat(Y).numpy(), torch.cat(P).numpy(), average="weighted")

    f32 = eval_serving_model(infer_cpu, sm_test)
    f8 = eval_serving_model(q_model, sm_test)
    cpu32 = next(r for r in bench_rows if r["device"] == "cpu" and r["batch"] == 1
                 and r["precision"] == "fp32")
    int8_tradeoff = {
        "cpu_fp32_weighted_f1": f32, "cpu_int8_weighted_f1": f8, "f1_delta": f8 - f32,
        "cpu_fp32_p50_ms": cpu32["p50_ms"], "cpu_int8_p50_ms": r["p50_ms"],
        "speedup": cpu32["p50_ms"] / r["p50_ms"],
    }
    print(f"\nint8 trade-off: F1 {f32:.4f} -> {f8:.4f} ({f8-f32:+.4f}), "
          f"latency {cpu32['p50_ms']:.2f} -> {r['p50_ms']:.2f} ms "
          f"({cpu32['p50_ms']/r['p50_ms']:.2f}x)")
except Exception as e:
    print("int8 quantization skipped:", type(e).__name__, e)

# ---- preprocessing cost ---------------------------------------------
t0 = time.perf_counter()
for r_ in sm_test[:1500]:
    feat.encode(r_["words"])
prep_ms = (time.perf_counter() - t0) * 1000.0 / 1500

print(f"\n{'device':>7} {'prec':>13} {'bs':>5} {'p50':>9} {'p95':>9} {'p99':>9} {'qps':>12}")
for r_ in bench_rows:
    print(f"{r_['device']:>7} {r_['precision']:>13} {r_['batch']:>5} "
          f"{r_['p50_ms']:>8.3f}ms {r_['p95_ms']:>8.3f}ms {r_['p99_ms']:>8.3f}ms "
          f"{r_['throughput_qps']:>12,.0f}")
print(f"\npreprocessing (featurize + tokenize), per sentence: {prep_ms:.4f} ms")

b1 = next((r_ for r_ in bench_rows if r_["batch"] == 1), None)
if b1:
    e2e = b1["p50_ms"] + prep_ms
    print(f"end-to-end p50 at batch 1 ({b1['device']}/{b1['precision']}): {e2e:.3f} ms "
          f"-> {'MEETS' if e2e < 10 else 'MISSES'} the PS single-digit-ms target")

model.to(DEVICE)

In [ ]:
# =====================================================================
# FIGURES FOR THE WHITEPAPER
# =====================================================================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16, 4))

lv = [r_["drift"] for r_ in drift_results]
for t in ACTIVE:
    ax[0].plot(lv, [r_[f"{t}_weighted_f1"] for r_ in drift_results], "o-", label=t)
ax[0].set_xlabel("spelling-perturbation strength"); ax[0].set_ylabel("weighted F1")
ax[0].set_title("Robustness on real test data"); ax[0].set_xticks(lv)
ax[0].grid(alpha=0.3); ax[0].legend()

src = [r_ for r_ in bench_rows if r_["device"] == "cuda" and r_["precision"] == "fp32"] or \
      [r_ for r_ in bench_rows if r_["device"] == "cpu" and r_["precision"] == "fp32"]
ax[1].plot([r_["batch"] for r_ in src], [r_["throughput_qps"] for r_ in src], "o-")
ax[1].set_xscale("log", base=2); ax[1].set_yscale("log")
ax[1].set_xlabel("batch size"); ax[1].set_ylabel("sentences/s")
ax[1].set_title("Throughput vs batch size"); ax[1].grid(alpha=0.3, which="both")

ax[2].plot([r_["seq_len"] for r_ in seq_rows], [r_["p50_ms"] for r_ in seq_rows], "o-", label="p50")
ax[2].plot([r_["seq_len"] for r_ in seq_rows], [r_["p99_ms"] for r_ in seq_rows], "s--", label="p99")
ax[2].axhline(10, color="r", ls=":", label="10 ms budget")
ax[2].set_xlabel("sequence length"); ax[2].set_ylabel("latency (ms), batch 1")
ax[2].set_title("Latency vs input length"); ax[2].grid(alpha=0.3); ax[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(cfg.out_dir, "results.png"), dpi=150)
plt.show()

---
## 9 · Export — reproducible training *and inference* codebase

Everything needed to reproduce and to serve lands in `out_dir`:

| File | What it is |
|---|---|
| `best.pt`, `pretrained.pt` | fine-tuned and MLM-only weights |
| `tokenizer.json` | the trained byte-level BPE, emoji tokens included |
| `config.json` | the exact config that produced the run (seeded) |
| `metrics.json` | every number in §4–§8, machine-readable |
| `results.png` | the three whitepaper figures |
| `inference.py` | standalone loader + `classify()`, no notebook required |

In [ ]:
# =====================================================================
# SAVE ARTIFACTS
# =====================================================================
from dataclasses import asdict

results = {
    "problem": "Inter IIT Bootcamp 2026 - PS 4: The Polyglot's Shorthand",
    "config": asdict(cfg),
    "environment": {"device": DEVICE, "torch": torch.__version__,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None},
    "data": {
        "sentimix": {"train": len(sm_train), "val": len(sm_val), "test": len(sm_test),
                     "unlabelled": len(sm_unlabelled)},
        "hinglid": {"train": len(lid_train), "val": len(lid_val), "test": len(lid_test)},
        "mlm_sentences": len(mlm_corpus), "vocab_size": VOCAB, "emoji_tokens": len(emoji_vocab),
    },
    "footprint": footprint,
    "mlm_history": mlm_history,
    "finetune_history": history,
    "test_clean": clean_metrics,
    "semeval_reference": SEMEVAL_REFERENCE,
    "drift_sweep": drift_results,
    "emoji_ablation": emoji_ablation,
    "error_analysis_slices": slice_report,
    "lid_error_analysis": lid_error,
    "benchmark": bench_rows,
    "seq_len_sweep": seq_rows,
    "int8_tradeoff": int8_tradeoff,
    "preprocess_ms_per_sentence": prep_ms,
    "sources": {
        "sentimix": "SemEval-2020 Task 9 (Patwa et al., 2020.semeval-1.100), CodaLab 20654",
        "hinglid": "L3Cube-HingLID (Nayak & Joshi, arXiv:2204.08398), CC BY-NC-SA 4.0",
    },
}

with open(os.path.join(cfg.out_dir, "metrics.json"), "w") as f:
    json.dump(results, f, indent=2, default=float)
with open(os.path.join(cfg.out_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

# standalone inference script: loads the artifacts and exposes classify()
INFERENCE_SRC = '''# Standalone inference for the Polyglot Shorthand encoder.
# Requires: torch, tokenizers, and normalization.py exported alongside this file.
import json, torch
from tokenizers import Tokenizer

ART = "."                     # directory holding best.pt / tokenizer.json / config.json

def load(device="cpu"):
    cfg = json.load(open(f"{ART}/config.json"))
    tok = Tokenizer.from_file(f"{ART}/tokenizer.json")
    state = torch.load(f"{ART}/best.pt", map_location=device)
    return cfg, tok, state

# Rebuild PolyglotForTasks with the classes from the notebook, load `state`,
# then reuse prep() / feat.encode() / classify() exactly as defined there.
'''
with open(os.path.join(cfg.out_dir, "inference.py"), "w") as f:
    f.write(INFERENCE_SRC)

print("saved to", cfg.out_dir)
for fn in sorted(os.listdir(cfg.out_dir)):
    p = os.path.join(cfg.out_dir, fn)
    if os.path.isfile(p):
        print(f"  {fn:<20} {os.path.getsize(p)/1e6:>8.2f} MB")

print("\n" + "=" * 66)
print("PS 4 SUMMARY")
print("=" * 66)
print(f"params                   : {footprint['total_params']:,}  "
      f"({footprint['budget_used_pct']:.3f}% of the 500M budget)")
print(f"fp32 / int8 size         : {footprint['fp32_mb']:.1f} MB / {footprint['int8_mb']:.1f} MB")
for k in sorted(clean_metrics):
    print(f"{k:<25}: {clean_metrics[k]:.4f}")
print(f"perturbation retention   : {100*worst/base:.1f}%  ({head}, level 3)")
if emoji_ablation:
    print(f"emoji subset F1          : {emoji_ablation['emoji_subset']:.4f} -> "
          f"{emoji_ablation['emoji_subset_stripped']:.4f} when stripped")
if lid_error:
    print(f"LID acc on unseen words  : {lid_error['acc_unseen_in_train']:.4f}")
if b1:
    print(f"batch-1 p50 model / e2e  : {b1['p50_ms']:.3f} ms / {b1['p50_ms']+prep_ms:.3f} ms "
          f"({b1['device']}/{b1['precision']})")
best_tp = max(bench_rows, key=lambda r_: r_["throughput_qps"])
print(f"peak throughput          : {best_tp['throughput_qps']:,.0f} sent/s @ bs={best_tp['batch']} "
      f"{best_tp['device']}/{best_tp['precision']}")
print("=" * 66)

---
## 10 · What to do next

Ordered by marks-per-hour for the whitepaper:

1. **Ablate the two input channels.** `model.encoder.use_skel = False` and
   `model.encoder.use_feats = False` are already wired in — retrain each variant and overlay the
   three drift curves from §7. That single figure turns "we designed for robustness" into a
   measured claim, and it's the most defensible thing in the whole submission.
2. **Ablate the drift augmentation.** Set `augment_drift_p = 0.0`, retrain, compare. Separates
   what the architecture buys from what the data augmentation buys — reviewers will ask.
3. **Baseline table against the models the PS names.** Fine-tune IndicBERT, mBERT and HingRoBERTa
   on this exact split and put weighted F1, parameter count, tokenizer fertility and batch-1
   latency in one table. *That table is the argument.* An encoder at 0.003% of the budget landing
   within a few F1 of a 270M model is the entire pitch, and fertility is where you show IndicBERT
   shattering Romanized input.
4. **More pretraining data.** L3Cube-HingCorpus is 52.93M Romanized sentences / 1.04B tokens, but
   it's a Google Drive file, not a raw URL, so it can't go in the download cell. Fetch it once,
   upload as a private Kaggle Dataset, attach it and point `mlm_corpus` at it. Going from 1.7M to
   even 100M tokens is the largest single accuracy lever here.
5. **Distillation.** Label unlabelled Romanized text with a frontier model, train this encoder on
   the soft labels. Directly answers the PS's "frontier LLMs can parse it but cost too much"
   framing.
6. **Generative tasks, if you want them.** The Note lets you pick tasks, so summarization and QA
   are optional. If you add them, bolt a 2–4 layer decoder onto this encoder rather than training
   a separate model — and be honest that no gold Romanized Hinglish summarization set exists.

**Limits to state in the writeup.** Pretraining here is ~1.7M tokens, so absolute accuracy will sit
below a HingBERT fine-tune and likely below the SemEval leaderboard top. What survives scrutiny is
the *relative* story: fertility, drift retention, emoji dependence, LID accuracy on unseen words,
parameter count and latency. The emoji subset is ~250 test rows, so quote that delta with its n.
The phonetic fold table is hand-built for Hindi Romanization and would need extending for Bangla,
Tamil or Urdu before claiming South-Asia-wide coverage.